# V5 — Extensión del universo de activos

## Qué decide este notebook

El estudio V4 usó **K = 5** activos y **~65 eventos por activo**. Contra la vara de una revista
de finanzas eso es un piloto, y arrastra tres problemas ligados:

| problema | por qué importa |
|---|---|
| K = 5 | los papers de *Quantitative Finance* usan decenas de activos |
| `ρ_naive ∈ [0.787, 0.890]` | **los datos no muestran la criticidad aparente que la teoría explica** |
| Nivel III nunca dispara | con 5 filas por columna, un testigo-cero separado es aritméticamente casi imposible |

Los tres tienen **un mismo arreglo**: más activos, incluyendo **small-caps deliberadamente**.
Una altcoin pequeña plausiblemente **no excita a BTC** — ahí están los ceros estructurales.
Y con más nodos y más co-excitación, `ρ(B̂)` debería subir hacia el régimen `n ≈ 1`.

## Las tres salidas posibles

Al final, la celda **VEREDICTO** imprime cuál de estos tres escenarios ocurrió:

- **A** — `ρ_naive → 1` **y** aparecen testigos-cero → paper fuerte para *Quantitative Finance*
- **B** — aparecen testigos-cero pero `ρ` no sube → paper de identificación / estructura de red
- **C** — ninguna de las dos → **la señal es que no es un paper de finanzas** → *Electronic Journal of Statistics* con el manuscrito completo

## Orden de ejecución

Las celdas 1–4 son **baratas y protegen las caras**. No saltes ninguna:

1. **Loader** — importa las funciones de tu V4 (no reescribe nada)
2. **Config**
3. **PREFLIGHT** — verifica que **todos** los ficheros existan en Binance *antes* de bajar nada,
   y estima el tamaño total de la descarga
4. **AUTO-TEST** — corre la cadena completa sobre datos sintéticos en ~20 s

Si 3 o 4 fallan, **no sigas**.


## 1 · Loader — importa las funciones del V4

No reescribe tu código: ejecuta las celdas de definición de tu notebook V4 y verifica que todo lo necesario quedó cargado.


In [ ]:
import json, ast, sys
from pathlib import Path

# --- localizar el notebook V4 ---
CANDS = sorted(Path.cwd().glob("*V4*CONFIRMATORY*.ipynb")) + \
        sorted(Path.cwd().glob("*crypto*V4*.ipynb")) + \
        sorted(Path.cwd().glob("*crypto_common_drive*.ipynb"))
V4 = CANDS[0] if CANDS else None
assert V4 is not None, (
    "No encuentro el notebook V4 en este directorio.\n"
    "Copia 'crypto_common_drive_event_network_V4_CONFIRMATORY.ipynb' junto a este notebook."
)
print(f"V4 encontrado: {V4.name}")

nb_v4 = json.load(open(V4))
code_cells = [(i, "".join(c["source"])) for i, c in enumerate(nb_v4["cells"])
              if c["cell_type"] == "code"]

def is_definition_cell(src):
    """Celda segura de ejecutar: define funciones y no llama nada a nivel superior."""
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return False
    has_def = any(isinstance(n, (ast.FunctionDef, ast.ClassDef)) for n in tree.body)
    top_calls = [n for n in tree.body
                 if isinstance(n, ast.Expr) and isinstance(n.value, ast.Call)]
    return has_def and not top_calls

# la primera celda de codigo trae imports + configuracion: la necesitamos
to_exec = [code_cells[0]] + [(i, s) for i, s in code_cells[1:] if is_definition_cell(s)]
print(f"ejecutando {len(to_exec)} celdas de definicion: {[i for i,_ in to_exec]}")

for i, src in to_exec:
    try:
        exec(compile(src, f"<V4 cell {i}>", "exec"), globals())
    except Exception as exc:
        print(f"  ! celda {i} fallo ({type(exc).__name__}: {exc}) — continuo")

import numpy as np

# ── PARCHE DE COMPATIBILIDAD (pandas copy-on-write) ─────────────────────────
# En pandas 2.x, df.to_numpy() puede devolver una vista de SOLO LECTURA, y
# first_pc_score del V4 hace 'X -= X.mean(...)' in situ -> ValueError.
# Se redefine con copia explicita; el resultado numerico es identico.
_ssd = safe_standardize_df
def first_pc_score(df):
    X = np.array(_ssd(df).to_numpy(dtype=float), copy=True)
    if X.shape[1] == 0:
        return np.zeros(X.shape[0])
    X = X - X.mean(axis=0, keepdims=True)
    _, _, vt = np.linalg.svd(X, full_matrices=False)
    score = X @ vt[0]
    agg = X.mean(axis=1)
    if np.corrcoef(score, agg)[0, 1] < 0:
        score = -score
    return (score - np.mean(score)) / (np.std(score) + 1e-12)
print("parche aplicado: first_pc_score copy-safe")

REQUIRED = [
    "download_one_aggtrade", "read_aggtrade_zip", "load_window_panel",
    "panel_to_wide_features", "build_events_from_features", "build_drive_matrix",
    "apply_drive_timing", "fit_network", "build_filtered_histories",
    "spectral_radius", "rank1_share", "zero_witness_diagnostics", "run_design",
    "aggtrade_url", "date_range_inclusive",
]
missing = [n for n in REQUIRED if n not in globals()]
assert not missing, f"*** FALTAN funciones del V4: {missing} — no sigas ***"
print(f"OK: las {len(REQUIRED)} funciones necesarias estan cargadas.")


## 2 · Configuración de la extensión

Los activos van **ordenados por capitalización**, de modo que tomar los primeros `K` da un
universo progresivamente más amplio y **cada tramo añade activos más pequeños**. Los últimos
son los candidatos a testigo-cero.


In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")

# ── universo ORDENADO por capitalizacion (tramos de 5) ──
SYMBOLS_TIERED = [
    # K=5  large caps  (el universo del V4)
    "BTCUSDT", "ETHUSDT", "BNBUSDT", "XRPUSDT", "SOLUSDT",
    # K=10
    "ADAUSDT", "DOGEUSDT", "MATICUSDT", "DOTUSDT", "LTCUSDT",
    # K=15
    "TRXUSDT", "AVAXUSDT", "LINKUSDT", "ATOMUSDT", "ETCUSDT",
    # K=20  mid caps
    "XLMUSDT", "BCHUSDT", "NEARUSDT", "FILUSDT", "ALGOUSDT",
    # K=25  mid/small
    "VETUSDT", "ICPUSDT", "HBARUSDT", "EOSUSDT", "SANDUSDT",
    # K=30  small caps  <- los candidatos a TESTIGO-CERO
    "MANAUSDT", "AXSUSDT", "CHZUSDT", "ENJUSDT", "ZILUSDT",
]
K_GRID = [5, 10, 15, 20, 25, 30]

# ── ventanas: 3 shocks + 3 controles ──
WINDOWS_EXT = [
    {"name": "Terra_Luna",         "kind": "shock",   "start": "2022-05-09", "end": "2022-05-12"},
    {"name": "FTX_collapse",       "kind": "shock",   "start": "2022-11-08", "end": "2022-11-10"},
    {"name": "Banking_crypto",     "kind": "shock",   "start": "2023-03-10", "end": "2023-03-13"},
    {"name": "Control_2022_Aug_A", "kind": "control", "start": "2022-08-08", "end": "2022-08-10"},
    {"name": "Control_2022_Oct",   "kind": "control", "start": "2022-10-10", "end": "2022-10-12"},
    {"name": "Control_2023_Feb",   "kind": "control", "start": "2023-02-06", "end": "2023-02-08"},
]

# ── umbral de eventos: bajarlo SUBE el tamano muestral efectivo ──
#    q=0.985 daba ~65 eventos/activo;  q=0.95 da ~3x mas.
Q_GRID = [0.95, 0.97, 0.985]

# ── Fase A (exploracion): inferencia barata, rejilla amplia ──
# CRITICO: zero_witness_diagnostics exige B_boot con >= 10 replicas; por debajo
# de eso pone se_B = NaN y TODAS las columnas devuelven passes=False SIN evaluar.
# Con 8 el test de Nivel III quedaria desactivado en silencio.
BOOTSTRAP_REPS = 24
NULL_REPS      = 12
MIN_EVENT_COUNT_PER_ASSET = 25      # mas exigente que el V4 (era 12)

DESIGN_BASE = dict(event_type="volume_burst", q=0.985, bin="1min",
                   lags=60, half_life=10, drive_mode="loo_market_activity",
                   drive_timing="lag1")

EXT = Path.cwd() / "v5_extension"
(EXT / "tables").mkdir(parents=True, exist_ok=True)
(EXT / "figures").mkdir(parents=True, exist_ok=True)

print(f"universo: {len(SYMBOLS_TIERED)} activos | ventanas: {len(WINDOWS_EXT)} | q: {Q_GRID}")
print(f"rejilla Fase A: {len(WINDOWS_EXT)} x {len(Q_GRID)} x {len(K_GRID)} = "
      f"{len(WINDOWS_EXT)*len(Q_GRID)*len(K_GRID)} disenos")


## 2b · Verificación del caché — ¿dónde están tus datos ya descargados?

Tu V4 guarda en **dos niveles**, ambos relativos a la carpeta desde la que ejecutas:

```
<carpeta actual>/crypto_common_drive_data_V3/raw_binance/   <- los .zip de Binance
<carpeta actual>/crypto_common_drive_outputs_V3/cache/      <- paneles por minuto (.parquet)
```

El parquet se consulta **primero**: si existe, ni se abre el zip.

**Por eso el notebook V5 debe ejecutarse desde la MISMA carpeta que el V4.** Si lo abres desde
otro sitio, `Path.cwd()` cambia, las rutas apuntan a otro lado y se descarga todo otra vez.

Esta celda te dice si el caché se encontró y cuántos ficheros faltan realmente.


In [ ]:
print(f"carpeta actual : {ROOT}")
print(f"zips (RAW)     : {RAW}")
print(f"parquets(CACHE): {CACHE}")
print()

if not RAW.exists() or not any(RAW.rglob("*.zip")):
    print("*" * 70)
    print("AVISO: no encuentro descargas previas en esta carpeta.")
    print("Si ya corriste el V4, cierra este notebook y abrelo desde la carpeta que")
    print("contiene 'crypto_common_drive_data_V3'. Si no, se descargara todo de nuevo.")
    print("*" * 70)
    print()

needed = [(s, d) for w in WINDOWS_EXT
          for d in date_range_inclusive(w["start"], w["end"])
          for s in SYMBOLS_TIERED]

have_pq  = sum(1 for s, d in needed if (CACHE / f"{s}_{d}_1min.parquet").exists())
have_zip = sum(1 for s, d in needed
               if (RAW / s / f"{s}-aggTrades-{d}.zip").exists()
               and (RAW / s / f"{s}-aggTrades-{d}.zip").stat().st_size > 100)
resuelto = sum(1 for s, d in needed
               if (CACHE / f"{s}_{d}_1min.parquet").exists()
               or ((RAW / s / f"{s}-aggTrades-{d}.zip").exists()
                   and (RAW / s / f"{s}-aggTrades-{d}.zip").stat().st_size > 100))

print(f"ficheros necesarios      : {len(needed)}")
print(f"  ya en parquet (rapido) : {have_pq}")
print(f"  ya en zip              : {have_zip}")
print(f"  FALTAN por descargar   : {len(needed) - resuelto}")
print()
print(f"total en disco: {len(list(RAW.rglob('*.zip')))} zips, "
      f"{len(list(CACHE.glob('*.parquet')))} parquets")

# el V4 lee estas dos banderas dentro de aggregate_symbol_day
RUN_DOWNLOAD = True        # permite bajar SOLO lo que falte
FORCE_REDOWNLOAD = False   # nunca re-descarga lo que ya existe
print("\nRUN_DOWNLOAD=True, FORCE_REDOWNLOAD=False -> solo se baja lo que falta.")


## 3 · PREFLIGHT — verificar la descarga **antes** de bajar nada

Comprueba por HTTP HEAD que **cada** fichero `(activo, fecha)` existe en Binance Vision, y usa
`Content-Length` para estimar el tamaño total. Descarta automáticamente los activos con cobertura
incompleta.

Esto tarda ~1–2 min y te evita descubrir a mitad de una descarga de varios GB que un par no
cotizaba en esa fecha.


In [ ]:
import urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor

def head_info(url, timeout=20):
    """(existe, bytes). HEAD no descarga el cuerpo."""
    req = urllib.request.Request(url, method="HEAD")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return True, int(r.headers.get("Content-Length", 0))
    except Exception:
        return False, 0

# todas las combinaciones (activo, fecha) requeridas
jobs = []
for w in WINDOWS_EXT:
    for d in date_range_inclusive(w["start"], w["end"]):
        for s in SYMBOLS_TIERED:
            jobs.append((s, d, aggtrade_url(s, d)))

print(f"verificando {len(jobs)} ficheros ...")
with ThreadPoolExecutor(max_workers=16) as pool:
    results = list(pool.map(lambda j: head_info(j[2]), jobs))

pf = pd.DataFrame([{"symbol": s, "date": d, "ok": ok, "bytes": b}
                   for (s, d, _), (ok, b) in zip(jobs, results)])
cov = pf.groupby("symbol").agg(cobertura=("ok", "mean"), MB=("bytes", lambda x: x.sum()/1e6))
cov = cov.reindex(SYMBOLS_TIERED)

SYMBOLS_OK = cov.index[cov.cobertura >= 0.999].tolist()
DROPPED    = cov.index[cov.cobertura < 0.999].tolist()

print(f"\ncompletos : {len(SYMBOLS_OK)}/{len(SYMBOLS_TIERED)}")
if DROPPED:
    print("DESCARTADOS (cobertura incompleta):")
    print(cov.loc[DROPPED, ["cobertura"]].round(3).to_string())
print(f"\ndescarga total estimada: {cov.loc[SYMBOLS_OK,'MB'].sum()/1000:.2f} GB")
print(f"espacio en disco necesario: ~{cov.loc[SYMBOLS_OK,'MB'].sum()/1000*1.1:.2f} GB")

# el orden por capitalizacion se conserva
SYMBOLS_TIERED = [s for s in SYMBOLS_TIERED if s in SYMBOLS_OK]
K_GRID = [k for k in K_GRID if k <= len(SYMBOLS_TIERED)]
if len(SYMBOLS_TIERED) not in K_GRID:
    K_GRID.append(len(SYMBOLS_TIERED))

assert len(SYMBOLS_TIERED) >= 15, (
    f"Solo {len(SYMBOLS_TIERED)} activos con cobertura completa. "
    "Con menos de 15 la extension no responde la pregunta — revisa las ventanas."
)
print(f"\nOK -> universo final: {len(SYMBOLS_TIERED)} activos | K_GRID = {K_GRID}")

## 4 · AUTO-TEST — la cadena completa sobre datos sintéticos (~20 s)

Fabrica un panel con **exactamente el esquema** que produce tu agregador, a partir de una red
**esparsa por bloques** (los large-caps se excitan entre sí; los small-caps **no** excitan a los
large-caps) más un drive común. Luego lo pasa por `run_design`.

Verifica tres cosas antes de tocar datos reales:

1. la cadena corre de extremo a extremo sin errores de esquema,
2. el deconfounding **deflacta** (`ρ_naive > ρ_drive`) y `ΔB` sale **de rango bajo**,
3. **los testigos-cero disparan** cuando la red realmente tiene ceros de bloque.

El punto 3 es el importante: confirma que si los datos reales tienen esa estructura, la
detectaríamos.


In [ ]:
def synthetic_panel(symbols, T=2000, n_large=10, seed=0):
    """Panel long-format con el esquema que espera panel_to_wide_features."""
    rng = np.random.default_rng(seed)
    K = len(symbols)
    idx = pd.date_range("2022-11-08", periods=T, freq="1min", tz="UTC")

    # red ESPARSA por bloques: small-caps NO excitan a large-caps -> ceros estructurales
    B0 = np.zeros((K, K))
    for i in range(K):
        for j in range(K):
            if i == j:
                continue
            large_i, large_j = i < n_large, j < n_large
            p = 0.55 if (large_i and large_j) else (0.35 if (not large_i and large_j) else 0.0)
            if rng.uniform() < p:
                B0[i, j] = rng.uniform(0.05, 0.30)
    sr = spectral_radius(B0)
    if sr > 1e-9:
        B0 *= 0.75 / sr

    # drive comun persistente + carga heterogenea
    m = np.zeros(T)
    for t in range(1, T):
        m[t] = 0.97 * m[t-1] + rng.normal(0, 1)
    m = (m - m.mean()) / (m.std() + 1e-9)
    c = np.concatenate([rng.uniform(0.7, 1.0, n_large),
                        rng.uniform(0.2, 0.5, K - n_large)])

    # intensidad = drive + propagacion por la red + ruido
    lat = np.zeros((T, K))
    for t in range(1, T):
        lat[t] = c * m[t] + lat[t-1] @ B0.T * 0.9 + rng.normal(0, 0.5, K)

    qv   = np.exp(1.5 + 0.8 * lat) * rng.lognormal(0, 0.25, (T, K))
    ntr  = np.maximum(1, (qv / 40.0)).astype(int)
    lr   = 0.0015 * lat + rng.normal(0, 0.0008, (T, K))
    imb  = np.tanh(0.4 * lat + rng.normal(0, 0.5, (T, K)))

    rows = []
    for k, s in enumerate(symbols):
        rows.append(pd.DataFrame({
            "time": idx, "symbol": s,
            "volume": qv[:, k] / 20000.0,
            "quote_volume": qv[:, k],
            "n_trades": ntr[:, k],
            "logret": lr[:, k],
            "abs_logret": np.abs(lr[:, k]),
            "imbalance": imb[:, k],
            "abs_imbalance_x_volume": np.abs(imb[:, k]) * qv[:, k],
        }))
    return pd.concat(rows, ignore_index=True), B0, c

# ── correr el auto-test ──
_syms = [f"T{i:02d}USDT" for i in range(20)]
_panel, _B0, _c = synthetic_panel(_syms, T=2000, n_large=10, seed=1)
_design = dict(DESIGN_BASE); _design["q"] = 0.95
_win = {"name": "SELFTEST", "kind": "control", "start": "2022-11-08", "end": "2022-11-10"}

_ok = True
try:
    _row, _det = run_design(_panel, _syms, _win, _design, seed_offset=0)
except Exception as exc:
    _ok = False
    print(f"*** LA CADENA FALLO: {type(exc).__name__}: {exc} ***")
    raise

zeros_true = 100 * (_B0 < 1e-12).mean()
print("AUTO-TEST — red sintetica esparsa por bloques, K=20, T=2000")
print(f"  ceros verdaderos en B0    : {zeros_true:.0f}%   (rho verdadero {spectral_radius(_B0):.3f})")
print(f"  activos retenidos         : {_row['K']}   eventos/activo (min) {_row['min_events_asset']}")
print(f"  rho_naive                 : {_row['rho_naive']:.4f}")
print(f"  rho_drive                 : {_row['rho_drive']:.4f}")
print(f"  delta_rho                 : {_row['delta_rho']:+.4f}   <- debe ser POSITIVO")
print(f"  rank1(Delta B)            : {_row['rank1_delta']:.3f}    <- debe ser ALTO")
print(f"  columnas con testigo-cero : {_row['zero_witness_pass_cols']}/{_row['K']}   <- deben DISPARAR")
print(f"  Algoritmo 1 aplicable     : {_row['alg1_applicable']}")

_pass = (_row["delta_rho"] > 0) and (_row["rank1_delta"] > 0.5) and (_row["zero_witness_pass_cols"] >= 1)
print()
print("*** AUTO-TEST OK — sigue adelante ***" if _pass else
      "*** AUTO-TEST FALLA — NO sigas, avisame antes de descargar ***")

## 5 · Descarga

Reutiliza tu `download_one_aggtrade`, que ya cachea: **si vuelves a correr, no re-descarga**.
El preflight ya garantizó que todos los ficheros existen.


In [ ]:
import time
t0 = time.time()
todo = [(s, d) for w in WINDOWS_EXT
        for d in date_range_inclusive(w["start"], w["end"])
        for s in SYMBOLS_TIERED]

# saltar lo que ya esta resuelto (parquet o zip valido)
def ya_esta(s, d):
    if (CACHE / f"{s}_{d}_1min.parquet").exists():
        return True
    z = RAW / s / f"{s}-aggTrades-{d}.zip"
    return z.exists() and z.stat().st_size > 100

pendientes = [(s, d) for s, d in todo if not ya_esta(s, d)]
print(f"{len(todo)} necesarios | {len(todo)-len(pendientes)} ya en cache | "
      f"{len(pendientes)} por descargar")

fails = []
for n, (s, d) in enumerate(pendientes, 1):
    if download_one_aggtrade(s, d, force=False) is None:
        fails.append((s, d))
    if n % 25 == 0 or n == len(pendientes):
        print(f"  {n}/{len(pendientes)}  ({time.time()-t0:.0f}s)  fallos: {len(fails)}")

print(f"\nlisto en {(time.time()-t0)/60:.1f} min | fallos: {len(fails)}")
if fails:
    print("  ", fails[:10])
assert len(fails) < 0.02 * max(len(pendientes), 1), "Demasiados fallos — revisa la conexion."


## 6 · Construir los paneles por ventana


In [ ]:
PANELS = {}
for w in WINDOWS_EXT:
    try:
        # OJO: la firma del V4 es load_window_panel(symbols, window, bin_size)
        #      -> los ACTIVOS van primero, la ventana segunda.
        p = load_window_panel(SYMBOLS_TIERED, w)
        if p is None or len(p) == 0:
            print(f"{w['name']:>20}: panel VACIO — se descarta")
            continue
        PANELS[w["name"]] = p
        print(f"{w['name']:>20}: {len(p):>8} filas | "
              f"{p['symbol'].nunique():>2} activos | "
              f"{p['time'].nunique():>5} minutos")
    except Exception as exc:
        print(f"{w['name']:>20}: FALLO — {type(exc).__name__}: {exc}")

assert len(PANELS) >= 2, (
    f"Solo {len(PANELS)} paneles construidos. Revisa que la descarga (celda anterior) "
    "haya terminado sin fallos."
)
print(f"\n{len(PANELS)} paneles listos.")


## 7 · FASE A — el experimento central: barrido en K y en q

Esta es la celda que contesta la pregunta. Para cada ventana, cada `q` y cada `K`, registra:

- **`ρ_naive`** — ¿sube hacia 1 al aumentar K?
- **`zero_witness_pass_cols`** — ¿aparecen testigos-cero al añadir small-caps?
- **`min_events_asset`** — ¿es adecuado el tamaño muestral efectivo?
- **`rank1_delta`** — ¿se mantiene la geometría de rango bajo?

Inferencia barata (`B=8`) porque aquí solo exploramos. La confirmatoria va en la Fase B.


In [ ]:
rowsA, detailsA = [], {}
total = len(PANELS) * len(Q_GRID) * len(K_GRID)
n = 0; t0 = time.time()

for wname, panel in PANELS.items():
    w = next(x for x in WINDOWS_EXT if x["name"] == wname)
    for q in Q_GRID:
        for K in K_GRID:
            n += 1
            syms = SYMBOLS_TIERED[:K]
            des = dict(DESIGN_BASE); des["q"] = q
            try:
                r, det = run_design(panel[panel["symbol"].isin(syms)], syms, w, des,
                                    seed_offset=n)
                r["K_requested"] = K
                rowsA.append(r)
                detailsA[(wname, q, K)] = det
            except Exception as exc:
                rowsA.append(dict(window=wname, q=q, K_requested=K,
                                  error=f"{type(exc).__name__}: {exc}"))
            if n % 10 == 0 or n == total:
                print(f"  {n}/{total}  ({time.time()-t0:.0f}s)")

A = pd.DataFrame(rowsA)
A.to_csv(EXT / "tables" / "v5_phaseA_sweep.csv", index=False)
ok = A[A.get("error").isna()] if "error" in A.columns else A
print(f"\nlisto: {len(ok)}/{len(A)} disenos OK, {(time.time()-t0)/60:.1f} min")

## 8 · ¿Sube `ρ` con K? ¿Disparan los testigos-cero?


In [ ]:
import matplotlib.pyplot as plt

piv_rho = ok.pivot_table(index="K_requested", columns="q", values="rho_naive", aggfunc="mean")
piv_zw  = ok.pivot_table(index="K_requested", columns="q", values="zero_witness_pass_cols", aggfunc="mean")
piv_ev  = ok.pivot_table(index="K_requested", columns="q", values="min_events_asset", aggfunc="mean")
piv_r1  = ok.pivot_table(index="K_requested", columns="q", values="rank1_delta", aggfunc="mean")

print("rho_naive medio (¿se acerca a 1?)");            print(piv_rho.round(3).to_string()); print()
print("columnas con testigo-cero (¿dispara Nivel III?)"); print(piv_zw.round(2).to_string()); print()
print("eventos minimos por activo (tamano muestral efectivo)"); print(piv_ev.round(0).to_string()); print()
print("rank1(Delta B)");                                print(piv_r1.round(3).to_string())

fig, ax = plt.subplots(1, 4, figsize=(17, 3.4))
for p, a, t in [(piv_rho, ax[0], r"$\rho_{naive}$ vs $K$"),
                (piv_zw,  ax[1], "columnas testigo-cero"),
                (piv_ev,  ax[2], "eventos min./activo"),
                (piv_r1,  ax[3], r"rank1($\Delta B$)")]:
    for c in p.columns:
        a.plot(p.index, p[c], "o-", label=f"q={c}")
    a.set_xlabel("K"); a.set_title(t, fontsize=10); a.legend(fontsize=7); a.grid(alpha=.3)
ax[0].axhline(1.0, color="r", ls="--", lw=1)
ax[1].axhline(1.0, color="r", ls="--", lw=1)
plt.tight_layout(); plt.savefig(EXT / "figures" / "v5_scaling.pdf", bbox_inches="tight")
plt.show()

## 9 · ¿Dispara el Nivel III? Y si dispara, ¿en qué columnas?

Si el Algoritmo 1 resulta aplicable, esto identifica **qué activos** aportaron el testigo-cero.
La predicción es que sean **small-caps** — que es justo el argumento económico: una altcoin
pequeña no excita a BTC.


In [ ]:
# El diagnostico del V4 usa errores estandar de BOOTSTRAP sobre un ajuste NNLS.
# En pruebas sinteticas con 76% de ceros REALES eso nunca dispara: NNLS deja las
# entradas exactamente en 0, las razones B/c son degeneradas y el umbral se traga
# todas las filas (candidate == n_valid, delta = NaN).
# Aqui se anade el diagnostico con errores ANALITICOS (la via del analisis sismico),
# que si produce separaciones finitas y por tanto es INTERPRETABLE.

def fit_analytic(Y, X):
    T_, p = X.shape
    A = np.column_stack([np.ones(T_), X])
    Gm = np.linalg.inv(A.T @ A + 1e-8 * np.eye(p + 1))
    Bf = Gm @ A.T @ Y
    r = Y - A @ Bf
    s2 = (r ** 2).sum(0) / max(T_ - p - 1, 1)
    return Bf[1:].T, np.sqrt(np.outer(s2, np.diag(Gm)[1:]))

def zw_analytic(B, SD, c, T_eff):
    K = B.shape[0]; c = np.asarray(c, float)
    Bp = np.maximum(B, 0.0)
    qT = np.sqrt(2 * np.log(max(3, K * T_eff)))
    ok = c > 1e-9
    if ok.sum() < 3:
        return pd.DataFrame(columns=["cand","n","delta","sep","passes"])
    out = []
    for k in range(K):
        r = Bp[ok, k] / c[ok]
        tau = np.maximum(SD[ok, k] / c[ok], 1e-12)
        lo = r.min()
        cand = r <= lo + 2 * qT * tau
        above = r[~cand]
        delta = float(above.min() - lo) if len(above) else np.nan
        sep = delta / (4 * qT * float(np.median(tau))) if np.isfinite(delta) else np.nan
        out.append((int(cand.sum()), int(ok.sum()), delta, sep,
                    bool(np.isfinite(sep) and sep > 1)))
    return pd.DataFrame(out, columns=["cand","n","delta","sep","passes"])

rows = []
for (wname, q, K), det in detailsA.items():
    Y = det["Y_df"].to_numpy(float)
    H = build_filtered_histories(Y, lags=DESIGN_BASE["lags"], half_life=DESIGN_BASE["half_life"])
    sl = slice(DESIGN_BASE["lags"], None)
    Bn, SD = fit_analytic(Y[sl], H[sl])
    chat = np.abs(np.asarray(det["drive"]["c"], float))
    chat = chat / max(chat.max(), 1e-12)
    tb = zw_analytic(Bn, SD, chat, len(Y[sl]))
    if not len(tb): continue
    rows.append(dict(window=wname, q=q, K=K,
                     swallow_all=int((tb.cand == tb.n).sum()), ncols=len(tb),
                     sep_med=float(tb.sep.median()), sep_max=float(tb.sep.max()),
                     passes=int(tb.passes.sum())))

Z = pd.DataFrame(rows)
if len(Z):
    Z.to_csv(EXT / "tables" / "v5_zero_witness_analytic.csv", index=False)
    print("DIAGNOSTICO DE TESTIGO-CERO con errores ANALITICOS\n")
    print(Z.groupby("K")[["sep_med","sep_max","passes","swallow_all"]].mean().round(3).to_string())
    print()
    print(f"columnas que PASAN (sep > 1) en total: {int(Z.passes.sum())}")
    print()
    print("Referencia sintetica (red con 76% de ceros REALES): sep ~ 0.50-0.61, 0 pasan.")
    print("Si tus datos reales dan sep del mismo orden, el limitante NO son los datos:")
    print("es que la condicion de separacion no se alcanza en este diseno.")
else:
    print("Sin diagnosticos calculables.")

# el diagnostico del V4 (bootstrap), para comparar
fired = ok[ok.get("alg1_applicable") == True] if "alg1_applicable" in ok.columns else ok.iloc[:0]
print(f"\n[V4, errores bootstrap] disenos con Alg.1 aplicable: {len(fired)}/{len(ok)}")


## 10 · La consecuencia económica: ¿cambia quién lidera el mercado?

Éste es el *"¿y qué?"* que un referee de finanzas va a exigir. Con K = 5 no había nada que
rankear; con K = 25–30 sí.

Comparamos la **in-strength** (cuánto recibe cada activo) y la **out-strength** (cuánto emite)
antes y después de quitar la actividad común. Si el ranking de quién dirige el mercado **cambia**,
eso es un resultado económico, no solo estadístico.


In [ ]:
from scipy.stats import spearmanr, kendalltau

rows = []
for (wname, q, K), det in detailsA.items():
    if K < 15:
        continue
    Bn, Bd = det["naive"]["B"], det["drive"]["B"]
    syms = det["symbols"]
    out_n, out_d = Bn.sum(axis=0), Bd.sum(axis=0)     # cuanto EMITE cada fuente
    in_n,  in_d  = Bn.sum(axis=1), Bd.sum(axis=1)     # cuanto RECIBE cada objetivo
    rows.append(dict(
        window=wname, q=q, K=K,
        spearman_out=spearmanr(out_n, out_d).correlation,
        spearman_in=spearmanr(in_n, in_d).correlation,
        kendall_out=kendalltau(out_n, out_d).correlation,
        top_emitter_naive=syms[int(np.argmax(out_n))],
        top_emitter_drive=syms[int(np.argmax(out_d))],
        top_changes=bool(np.argmax(out_n) != np.argmax(out_d)),
        n_rank_swaps=int(np.sum(np.argsort(-out_n) != np.argsort(-out_d))),
    ))

E = pd.DataFrame(rows)
if len(E):
    E.to_csv(EXT / "tables" / "v5_economic_consequence.csv", index=False)
    print(E.round(3).to_string(index=False))
    print()
    print(f"El emisor dominante CAMBIA en {E.top_changes.sum()}/{len(E)} disenos")
    print(f"Spearman medio del ranking de out-strength: {E.spearman_out.mean():.3f}")
    print(f"   (cerca de 1 = el deconfounding no altera el orden;")
    print(f"    bastante por debajo de 1 = ALTERA quien dirige el mercado -> resultado economico)")
else:
    print("Sin disenos con K>=15.")

## 11 · FASE B — inferencia confirmatoria sobre los mejores diseños

Solo ahora, y solo sobre los diseños seleccionados: `B = 500` bootstrap y `N = 500` nulo.
Selección **por diagnósticos**, nunca por `ρ`.


In [ ]:
BOOTSTRAP_REPS = 500
NULL_REPS      = 500

sel = ok.copy()
sel = sel[sel.get("min_events_asset", 0) >= 40]                 # tamano muestral adecuado
sel = sel[sel.get("spike_ratio", 0) >= 2.0]                     # Thm 12(ii)
sel = sel.sort_values(["zero_witness_pass_cols", "K_requested"], ascending=False)
sel = sel.head(8)

print(f"{len(sel)} disenos a confirmar (criterio: eventos>=40 y spike>=2)\n")
rowsB = []
for i, r in enumerate(sel.itertuples(), 1):
    w = next(x for x in WINDOWS_EXT if x["name"] == r.window)
    K = int(r.K_requested); syms = SYMBOLS_TIERED[:K]
    des = dict(DESIGN_BASE); des["q"] = r.q
    panel = PANELS[r.window]
    try:
        rb, _ = run_design(panel[panel["symbol"].isin(syms)], syms, w, des,
                           seed_offset=5000 + i)
        rb["K_requested"] = K
        rowsB.append(rb)
        print(f"  {i}/{len(sel)}  {r.window[:18]:>18} q={r.q} K={K:>2} | "
              f"drho={rb['delta_rho']:+.4f} rank1={rb['rank1_delta']:.3f} "
              f"nullp={rb['null_p_ge_observed']:.4f}")
    except Exception as exc:
        print(f"  {i}/{len(sel)}  FALLO: {exc}")

B = pd.DataFrame(rowsB)
if len(B):
    B.to_csv(EXT / "tables" / "v5_phaseB_confirmatory.csv", index=False)

## 12 · VEREDICTO — qué escenario ocurrió y qué revista

Esta celda decide la estrategia editorial con los criterios fijados **de antemano**.


In [ ]:
rho_max     = ok["rho_naive"].max()
rho_at_maxK = ok[ok.K_requested == ok.K_requested.max()]["rho_naive"].mean()
rho_at_5    = ok[ok.K_requested == 5]["rho_naive"].mean() if (ok.K_requested == 5).any() else np.nan
zw_fires    = int((ok.get("zero_witness_pass_cols", pd.Series([0])) >= 1).sum())
sep_ref     = Z.sep_max.max() if ("Z" in dir() and len(Z)) else np.nan
alg1_fires  = int(ok.get("alg1_applicable", pd.Series([False])).sum())
rank1_med   = ok["rank1_delta"].median()
econ        = E.top_changes.mean() if 'E' in dir() and len(E) else np.nan

print("="*72)
print("RESULTADOS")
print("="*72)
print(f"  rho_naive con K=5           : {rho_at_5:.3f}      (era 0.787-0.890 en el V4)")
print(f"  rho_naive con K={int(ok.K_requested.max()):<3}         : {rho_at_maxK:.3f}")
print(f"  rho_naive maximo            : {rho_max:.3f}")
print(f"  disenos con testigo-cero    : {zw_fires}/{len(ok)}")
print(f"  disenos con Alg.1 aplicable : {alg1_fires}/{len(ok)}")
print(f"  rank1(Delta B) mediano      : {rank1_med:.3f}")
if not np.isnan(sep_ref):
    print(f"  separacion maxima (analitica): {sep_ref:.3f}   (referencia sintetica ~0.6; hace falta >1)")
if not np.isnan(econ):
    print(f"  cambia el emisor dominante  : {100*econ:.0f}% de los disenos")

RHO_RISES = (rho_at_maxK > rho_at_5 + 0.03) or (rho_max >= 0.97)
ZW_FIRES  = (alg1_fires >= 3) or (("Z" in dir()) and len(Z) and Z.passes.sum() >= 3)

print()
print("="*72)
if RHO_RISES and ZW_FIRES:
    print("ESCENARIO A  ->  QUANTITATIVE FINANCE")
    print("="*72)
    print("""
  rho sube hacia el regimen critico Y los testigos-cero disparan.
  Tienes el paper completo: el fenomeno DEMOSTRADO en los datos, el mecanismo
  MEDIDO (rank1), y una red CORREGIDA. Se divide el manuscrito y se envia a QF.""")
elif ZW_FIRES:
    print("ESCENARIO B  ->  identificacion / estructura de red")
    print("="*72)
    print("""
  Los testigos-cero disparan pero rho no alcanza el regimen critico.
  Paper solido centrado en identificacion puntual y recuperacion de la red.
  Destinos: Quantitative Finance (con el enfasis en estructura, no en criticidad),
  Journal of Financial Econometrics, o Electronic Journal of Statistics.""")
elif RHO_RISES:
    print("ESCENARIO A-  ->  QF, pero sin identificacion puntual")
    print("="*72)
    print("""
  El fenomeno esta demostrado pero el Nivel III sigue sin alcanzarse.
  Sigue siendo enviable a QF: mecanismo + medicion de geometria + abstencion
  fundamentada. Mas debil que A, mas fuerte que el V4.""")
else:
    print("ESCENARIO C  ->  ELECTRONIC JOURNAL OF STATISTICS")
    print("="*72)
    print("""
  Ni rho sube ni aparecen testigos-cero, ni siquiera con small-caps.
  La senal es clara: NO es un paper de finanzas. No dividas el manuscrito.
  Envia la version completa (26 pp) a EJS, donde el estilo actual es nativo
  y el desk-reject por densidad tecnica no ocurre.""")
print("="*72)

summary = dict(rho_at_5=rho_at_5, rho_at_maxK=rho_at_maxK, rho_max=rho_max,
               zw_fires=zw_fires, alg1_fires=alg1_fires, rank1_med=rank1_med,
               econ_top_changes=econ, n_designs=len(ok))
pd.Series(summary).to_csv(EXT / "tables" / "v5_verdict.csv")
print(f"\nguardado en {EXT/'tables'}")
print("Mandame v5_phaseA_sweep.csv, v5_phaseB_confirmatory.csv y v5_verdict.csv")
